# 02 — Divisão Treino / Validação / Teste e Balanceamento

Divide os espectrogramas válidos por paciente (`70%` treino, `15%` validação, `15%` teste) evitando vazamento de dados (*data leakage*), aplica *oversampling* nas classes minoritárias do treino e salva `data/annotations/splits.csv`.

In [ ]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
ANNOT_PATH   = PROJECT_ROOT / 'data' / 'annotations' / 'annotations.csv'
OUT_PATH     = PROJECT_ROOT / 'data' / 'annotations' / 'splits.csv'

RATIOS = {'treino': 0.70, 'validacao': 0.15, 'teste': 0.15}
SEED   = 42

# 1. Carrega imagens validas
df = pd.read_csv(ANNOT_PATH)
df = df[(df.status == 'ok') & df.spectrogram_path.notna() & (df.patient_id != 'PN10')].copy()

# 2. Aloca cada paciente inteiro em um unico conjunto
counts = df.patient_id.value_counts()
total  = counts.sum()
filled = {k: 0 for k in RATIOS}
patient_set = {}

for pid, n in counts.items():
    target = max(RATIOS, key=lambda k: RATIOS[k] * total - filled[k])
    patient_set[pid] = target
    filled[target] += n

df['conjunto'] = df.patient_id.map(patient_set)
assert df.groupby('patient_id').conjunto.nunique().max() == 1, 'Vazamento de paciente detectado!'

In [ ]:
# 3. Oversampling apenas no conjunto de treino e exportacao do splits.csv
train = df[df.conjunto == 'treino']
other = df[df.conjunto != 'treino']

max_n = train['class'].value_counts().max()
train_bal = pd.concat([
    pd.concat([g, g.sample(max_n - len(g), replace=True, random_state=SEED)]) if len(g) < max_n else g
    for _, g in train.groupby('class')
])

out = pd.concat([train_bal, other], ignore_index=True)
out = pd.DataFrame({
    'id'      : range(1, len(out) + 1),
    'paciente': out.patient_id,
    'caminho' : out.spectrogram_path,
    'classe'  : out['class'].str.upper(),
    'conjunto': out.conjunto,
})
out.to_csv(OUT_PATH, index=False)

print(f'Salvo em: {OUT_PATH} ({len(out)} registros)')
display(out.groupby(['conjunto', 'classe']).size().unstack(fill_value=0))
display(out.head())